In [0]:
from pyspark.sql.window import Window 
from pyspark.sql.functions import trim
import pyspark.sql.functions as f

df= spark.read.table("dwh_project.bronze.cust_info")

###Trim all the spaces form strings

In [0]:

for c_name,data_type in df.dtypes:
    if data_type == "string" :
        df=df.withColumn(c_name,trim(c_name))


### normalization 

In [0]:


df=(
    df.withColumn(
        "cst_marital_status",
                  f.when(f.upper(f.col("cst_marital_status"))=='M',"married")
                  .when(f.upper(f.col("cst_marital_status"))=='S',"single")
                  .otherwise("n/a")
).withColumn(
        "cst_gndr",
        f.when(f.upper(f.col("cst_gndr"))=="M","male")
        .when(f.upper(f.col("cst_gndr"))=="F","femail")
        .otherwise("n/a"))
)


In [0]:

logic=(
    Window.partitionBy(f.col("cst_id"))
    .orderBy(f.col("cst_create_date"))
)
df=(
    df.withColumn("rank",f.row_number().over(logic))
    .filter((f.col("rank")==1) & (f.col("cst_id").isNotNull() ) )
    .drop("rank")
)

In [0]:
df.write.mode("overwrite").saveAsTable("dwh_project.silver.cust_info")